In [ ]:
# ═══════════════════════════════════════════════════════════════
# Verify: Load model and run inference (same as backend will do)
# ═══════════════════════════════════════════════════════════════

loaded = joblib.load(MODEL_PATH)
model = loaded["model"]
le_loaded = loaded["label_encoder"]

# Test predictions
test_cases = [
    {"name": "Light warm skin", "ITA": 65.0, "a_cie": 5.0, "b_cie": 18.0, "L_cie": 72.0, "warmth": 14.75},
    {"name": "Medium cool skin", "ITA": 35.0, "a_cie": -3.0, "b_cie": -8.0, "L_cie": 45.0, "warmth": -6.75},
    {"name": "Dark warm skin", "ITA": 5.0, "a_cie": 8.0, "b_cie": 10.0, "L_cie": 28.0, "warmth": 9.5},
    {"name": "Boundary case", "ITA": 45.0, "a_cie": 0.3, "b_cie": 0.2, "L_cie": 55.0, "warmth": 0.225},
]

print("Model verification — test predictions:")
print(f"  {'Input':<22} {'Predicted Season':<22} {'Prob':<8} {'2nd Season':<22} {'2nd Prob'}")
print(f"  {'-'*22} {'-'*22} {'-'*8} {'-'*22} {'-'*8}")

for tc in test_cases:
    features = np.array([[tc["ITA"], tc["a_cie"], tc["b_cie"], tc["L_cie"], tc["warmth"]]])
    proba = model.predict_proba(features)[0]
    top2_idx = np.argsort(proba)[::-1][:2]

    s1 = le_loaded.classes_[top2_idx[0]]
    p1 = proba[top2_idx[0]]
    s2 = le_loaded.classes_[top2_idx[1]]
    p2 = proba[top2_idx[1]]

    print(f"  {tc['name']:<22} {s1:<22} {p1*100:5.1f}%  {s2:<22} {p2*100:5.1f}%")

print(f"\nModel info: {loaded['model_name']}, {loaded['test_accuracy']*100:.1f}% accuracy")
print(f"Feature names: {loaded['feature_names']}")
print(f"Classes: {loaded['class_names']}")
print("\n--- DONE! Place season_classifier.joblib in backend/models/ ---")

## Step 10: Verify the Exported Model Loads Correctly

Quick sanity check — load the model and run inference.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Download the model file
# ═══════════════════════════════════════════════════════════════

from google.colab import files

print("Downloading season_classifier.joblib...")
print("Place this file in: backend/models/season_classifier.joblib")
files.download(MODEL_PATH)

print("\nDownloading training_features.csv...")
files.download(CSV_PATH)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Export the best model
# ═══════════════════════════════════════════════════════════════

export_data = {
    "model": best["model"],              # Full sklearn Pipeline (scaler + classifier)
    "label_encoder": le,                   # LabelEncoder (season names ↔ indices)
    "feature_names": FEATURE_COLS,         # ["ITA", "a_cie", "b_cie", "L_cie", "warmth"]
    "class_names": list(le.classes_),      # 12 season names
    "model_name": best_name,               # Which model type won
    "test_accuracy": best["test_acc"],
    "top2_accuracy": best["top2_acc"],
    "test_f1": best["test_f1"],
    "n_training_samples": len(X_train),
    "n_test_samples": len(X_test),
    "confidence_threshold": CONFIDENCE_THRESHOLD,
}

MODEL_PATH = "/content/season_classifier.joblib"
joblib.dump(export_data, MODEL_PATH, compress=3)

file_size = os.path.getsize(MODEL_PATH) / 1024
print(f"Model exported to: {MODEL_PATH}")
print(f"File size: {file_size:.0f} KB")
print(f"Model type: {best_name}")
print(f"Test accuracy: {best['test_acc']*100:.1f}%")
print(f"Top-2 accuracy: {best['top2_acc']*100:.1f}%")
print(f"Classes: {list(le.classes_)}")

# Also save the training data CSV for reference
CSV_PATH = "/content/training_features.csv"
df_confident.to_csv(CSV_PATH, index=False)
print(f"\nTraining data saved to: {CSV_PATH} ({len(df_confident)} rows)")

## Step 9: Export Model + Download

Exports the best model as `.joblib` with the label encoder and feature names. **Download this file and place it in `backend/models/`**.

In [ ]:
# =====================================================================
# Download UTKFace from Kaggle (most reliable public source)
# =====================================================================
# UTKFace: ~20K face images labeled [age]_[gender]_[race]_[date].jpg
# Race: 0=White, 1=Black, 2=Asian, 3=Indian, 4=Others
# =====================================================================

DATA_DIR = Path("/content/datasets")
DATA_DIR.mkdir(exist_ok=True)
UTKFACE_DIR = DATA_DIR / "UTKFace"

downloaded = False

if not UTKFACE_DIR.exists():
    try:
        kaggle_json = Path("/root/.kaggle/kaggle.json")
        if not kaggle_json.exists():
            print("=" * 60)
            print("  KAGGLE API KEY NEEDED (one-time setup)")
            print("=" * 60)
            print()
            print("1. Go to https://www.kaggle.com/settings")
            print("2. Scroll to API -> Create New Token (downloads kaggle.json)")
            print("3. Run this in a NEW cell first:")
            print()
            print("   from google.colab import files")
            print("   uploaded = files.upload()  # select kaggle.json")
            print("   !mkdir -p /root/.kaggle")
            print("   !mv kaggle.json /root/.kaggle/")
            print("   !chmod 600 /root/.kaggle/kaggle.json")
            print()
            print("4. Then re-run THIS cell")

            # Try interactive upload
            try:
                from google.colab import files as colab_files
                print("
OR upload kaggle.json now:")
                uploaded = colab_files.upload()
                if "kaggle.json" in uploaded:
                    !mkdir -p /root/.kaggle
                    with open("/root/.kaggle/kaggle.json", "wb") as f:
                        f.write(uploaded["kaggle.json"])
                    !chmod 600 /root/.kaggle/kaggle.json
                    print("kaggle.json saved!")
            except:
                pass

        if Path("/root/.kaggle/kaggle.json").exists():
            print("Downloading UTKFace from Kaggle...")
            !kaggle datasets download -d jangedoo/utkface-new -p /content/datasets/ --unzip

            import shutil
            for candidate in [
                DATA_DIR / "utkface_aligned_cropped" / "UTKFace",
                DATA_DIR / "UTKFace",
                DATA_DIR / "utkface-new",
                DATA_DIR / "crop_part1",
            ]:
                if candidate.exists() and list(candidate.glob("*.jpg")):
                    if candidate != UTKFACE_DIR:
                        candidate.rename(UTKFACE_DIR)
                    downloaded = True
                    break

            if not downloaded:
                UTKFACE_DIR.mkdir(exist_ok=True)
                for jpg in list(DATA_DIR.rglob("*.jpg"))[:20000]:
                    target = UTKFACE_DIR / jpg.name
                    if not target.exists():
                        shutil.copy2(str(jpg), str(target))
                if list(UTKFACE_DIR.glob("*.jpg")):
                    downloaded = True

    except Exception as e:
        print(f"Download failed: {e}")

    if not downloaded:
        UTKFACE_DIR.mkdir(exist_ok=True)
        print()
        print("MANUAL: Download from https://www.kaggle.com/datasets/jangedoo/utkface-new")
        print("Upload to /content/datasets/UTKFace/ via Colab sidebar")
else:
    print(f"UTKFace already at {UTKFACE_DIR}")
    downloaded = True

images = list(UTKFACE_DIR.glob("*.jpg")) + list(UTKFACE_DIR.glob("*.png"))
print(f"
Found {len(images)} face images")
if len(images) == 0:
    print("No images yet. Follow instructions above, then re-run.")

## Step 8: Compare Trained Model vs Old Lookup Table

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Detailed evaluation of the best model
# ═══════════════════════════════════════════════════════════════

y_pred_best = best["y_pred"]
y_proba_best = best["y_proba"]

# Classification report
present_labels = sorted(set(y_test) | set(y_pred_best))
present_names = [le.classes_[i] for i in present_labels]

print(f"\n{'='*60}")
print(f"  CLASSIFICATION REPORT — {best_name}")
print(f"{'='*60}\n")
print(classification_report(
    y_test, y_pred_best,
    labels=present_labels,
    target_names=present_names,
    zero_division=0
))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_best, labels=present_labels)

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=present_names,
            yticklabels=present_names, ax=ax)
ax.set_xlabel("Predicted Season", fontsize=12)
ax.set_ylabel("True Season", fontsize=12)
ax.set_title(f"Confusion Matrix — {best_name}\n"
             f"Accuracy: {best['test_acc']*100:.1f}% | Top-2: {best['top2_acc']*100:.1f}%",
             fontsize=14)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Per-ethnicity accuracy (if UTKFace)
if "ethnicity" in df_confident.columns:
    print(f"\n{'='*60}")
    print("  ACCURACY BY ETHNICITY (Fairness Check)")
    print(f"{'='*60}")
    eth_map = {0: "White", 1: "Black", 2: "Asian", 3: "Indian", 4: "Other"}

    test_df = df_confident.iloc[X_test.shape[0] * -1:].copy() if len(df_confident) > len(X_test) else df_confident.copy()

    # Re-predict on full dataset for ethnicity breakdown
    y_all_pred = best["model"].predict(X)
    df_confident["predicted"] = le.inverse_transform(y_all_pred)
    df_confident["correct"] = df_confident["season"] == df_confident["predicted"]

    for eth_code, eth_name in eth_map.items():
        subset = df_confident[df_confident["ethnicity"] == eth_code]
        if len(subset) > 0:
            acc = subset["correct"].mean() * 100
            print(f"  {eth_name:<10}: {acc:5.1f}% ({len(subset)} samples)")

## Step 7: Confusion Matrix + Classification Report

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, top_k_accuracy_score
)
import joblib

# ═══════════════════════════════════════════════════════════════
# Prepare data
# ═══════════════════════════════════════════════════════════════

FEATURE_COLS = ["ITA", "a_cie", "b_cie", "L_cie", "warmth"]

X = df_confident[FEATURE_COLS].values
le = LabelEncoder()
y = le.fit_transform(df_confident["season"].values)

print(f"Training data: {X.shape[0]} samples, {X.shape[1]} features, {len(le.classes_)} classes")
print(f"Classes: {list(le.classes_)}")

# Train/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

# ═══════════════════════════════════════════════════════════════
# Define models
# ═══════════════════════════════════════════════════════════════

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=2000, C=1.0, multi_class="multinomial",
            solver="lbfgs", random_state=42
        ))
    ]),
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(
            n_estimators=300, max_depth=15, min_samples_leaf=5,
            random_state=42, n_jobs=-1
        ))
    ]),
    "Gradient Boosting": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", GradientBoostingClassifier(
            n_estimators=200, max_depth=5, learning_rate=0.1,
            random_state=42
        ))
    ]),
    "MLP (Neural Network)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", MLPClassifier(
            hidden_layer_sizes=(128, 64, 32),
            activation="relu", max_iter=1000,
            early_stopping=True, validation_fraction=0.15,
            random_state=42
        ))
    ]),
}

# ═══════════════════════════════════════════════════════════════
# Cross-validation + Evaluation
# ═══════════════════════════════════════════════════════════════

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print("\n" + "=" * 60)
print("  MODEL COMPARISON (5-Fold Cross-Validation)")
print("=" * 60)

for name, model in models.items():
    # Cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)

    # Train on full training set, evaluate on test set
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    test_acc = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred, average="weighted")

    # Top-2 accuracy (correct season in top 2 predictions)
    top2_acc = top_k_accuracy_score(y_test, y_proba, k=2, labels=range(len(le.classes_)))

    results[name] = {
        "cv_mean": cv_scores.mean(),
        "cv_std": cv_scores.std(),
        "test_acc": test_acc,
        "test_f1": test_f1,
        "top2_acc": top2_acc,
        "model": model,
        "y_pred": y_pred,
        "y_proba": y_proba,
    }

    print(f"\n  {name}:")
    print(f"    CV Accuracy:   {cv_scores.mean()*100:.1f}% (+/- {cv_scores.std()*100:.1f}%)")
    print(f"    Test Accuracy: {test_acc*100:.1f}%")
    print(f"    Top-2 Accuracy:{top2_acc*100:.1f}%")
    print(f"    Weighted F1:   {test_f1*100:.1f}%")

# ═══════════════════════════════════════════════════════════════
# Pick the best model
# ═══════════════════════════════════════════════════════════════

best_name = max(results, key=lambda k: results[k]["test_acc"])
best = results[best_name]

print("\n" + "=" * 60)
print(f"  BEST MODEL: {best_name}")
print(f"  Test Accuracy: {best['test_acc']*100:.1f}%")
print(f"  Top-2 Accuracy: {best['top2_acc']*100:.1f}%")
print("=" * 60)

## Step 6: Train 3 Classifiers + Evaluate

We train Logistic Regression, Random Forest, and MLP — then compare with cross-validation.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEASON_MAP — The exact same lookup table from your backend
# ═══════════════════════════════════════════════════════════════

SEASON_MAP = [
    (55,  90, "warm", "Light Spring"),
    (41,  55, "warm", "True Spring"),
    (28,  41, "warm", "Warm Spring"),
    (55,  90, "cool", "Light Summer"),
    (41,  55, "cool", "True Summer"),
    (28,  41, "cool", "Soft Summer"),
    (10,  28, "cool", "Soft Autumn"),
    (10,  28, "warm", "True Autumn"),
    (-30, 10, "warm", "Deep Autumn"),
    (-30, 10, "cool", "Deep Winter"),
    (-55, -30, "cool", "True Winter"),
    (-55, -30, None,  "Bright Winter"),
]

CLASS_NAMES = [
    "Light Spring", "True Spring", "Warm Spring",
    "Light Summer", "True Summer", "Soft Summer",
    "Soft Autumn", "True Autumn", "Deep Autumn",
    "Deep Winter", "True Winter", "Bright Winter",
]

def assign_season_label(ita, a_cie, b_cie):
    """Assign a season using the old lookup table (for pseudo-labels)."""
    warmth = (0.25 * a_cie) + (0.75 * b_cie)
    tone = "warm" if warmth > 0 else "cool"

    for ita_min, ita_max, req_tone, season in SEASON_MAP:
        if ita_min <= ita < ita_max:
            if req_tone is None or req_tone == tone:
                return season

    if ita >= 90:
        return "Light Spring"
    return "Deep Winter"


# Assign pseudo-labels
df["season"] = df.apply(lambda r: assign_season_label(r["ITA"], r["a_cie"], r["b_cie"]), axis=1)

# Filter for high-confidence samples only (pseudo-label quality gate)
CONFIDENCE_THRESHOLD = 0.5
df_confident = df[df["confidence"] >= CONFIDENCE_THRESHOLD].copy()

print(f"Total samples: {len(df)}")
print(f"High-confidence samples (conf >= {CONFIDENCE_THRESHOLD}): {len(df_confident)}")
print(f"\nSeason distribution:")
print(df_confident["season"].value_counts().to_string())

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ITA distribution
axes[0].hist(df_confident["ITA"], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("ITA Angle")
axes[0].set_ylabel("Count")
axes[0].set_title("ITA Distribution")

# a* vs b* scatter colored by season
season_to_idx = {s: i for i, s in enumerate(CLASS_NAMES)}
colors = df_confident["season"].map(season_to_idx)
scatter = axes[1].scatter(df_confident["a_cie"], df_confident["b_cie"],
                          c=colors, cmap="Set3", alpha=0.5, s=10)
axes[1].set_xlabel("a* (green←→red)")
axes[1].set_ylabel("b* (blue←→yellow)")
axes[1].set_title("a* vs b* by Season")

# Ethnicity distribution
eth_labels = {0: "White", 1: "Black", 2: "Asian", 3: "Indian", 4: "Other", -1: "Unknown"}
df_confident["eth_name"] = df_confident["ethnicity"].map(eth_labels)
df_confident["eth_name"].value_counts().plot.bar(ax=axes[2])
axes[2].set_title("Ethnicity Distribution")
axes[2].set_ylabel("Count")

plt.tight_layout()
plt.show()

## Step 5: Generate Pseudo-Labels (Bootstrap from SEASON_MAP)

We use the current hardcoded rules to label each face, then filter for high-confidence results. This gives us a starting dataset to train a model that will **learn smoother, data-driven boundaries**.

In [ ]:
# Find all face images
FACE_DIR = UTKFACE_DIR if UTKFACE_DIR.exists() else DATA_DIR / "faces"
all_images = sorted(list(FACE_DIR.glob("*.jpg")) + list(FACE_DIR.glob("*.png")))
print(f"Found {len(all_images)} images in {FACE_DIR}")

# Limit for speed (increase for better training)
MAX_IMAGES = min(5000, len(all_images))
sample_images = all_images[:MAX_IMAGES]
print(f"Processing {MAX_IMAGES} images...")

# Extract features
records = []
failed = 0

for img_path in tqdm(sample_images, desc="Extracting features"):
    img = cv2.imread(str(img_path))
    if img is None:
        failed += 1
        continue

    features = extract_features(img)
    if features is None:
        failed += 1
        continue

    # Parse UTKFace metadata if available (age_gender_race_date.jpg)
    fname = img_path.stem
    parts = fname.split("_")
    ethnicity = int(parts[2]) if len(parts) >= 3 and parts[2].isdigit() else -1

    features["file"] = img_path.name
    features["ethnicity"] = ethnicity  # 0=White, 1=Black, 2=Asian, 3=Indian, 4=Other
    records.append(features)

df = pd.DataFrame(records)
print(f"\nExtracted features from {len(df)} images ({failed} failed)")
print(f"\nFeature statistics:")
print(df[["ITA", "a_cie", "b_cie", "L_cie", "warmth", "confidence"]].describe().round(2))

## Step 4: Extract Features from All Images

This runs the Lumiqe pipeline on every face image and builds a feature table.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Lumiqe Color Science — Feature Extraction
# (Identical to backend/app/cv/color_analysis.py)
# ═══════════════════════════════════════════════════════════════

from sklearn.cluster import KMeans

# --- Grey World Color Constancy (skin-masked) ---
def apply_grey_world(face_bgr, skin_mask=None):
    CLAMP_LO, CLAMP_HI = 0.85, 1.15
    if skin_mask is not None:
        mask_bool = skin_mask == 255
        if mask_bool.sum() > 100:
            skin_pixels = face_bgr[mask_bool]
            mean_b, mean_g, mean_r = np.mean(skin_pixels[:, 0]), np.mean(skin_pixels[:, 1]), np.mean(skin_pixels[:, 2])
        else:
            mean_b, mean_g, mean_r = np.mean(face_bgr[:,:,0]), np.mean(face_bgr[:,:,1]), np.mean(face_bgr[:,:,2])
    else:
        mean_b, mean_g, mean_r = np.mean(face_bgr[:,:,0]), np.mean(face_bgr[:,:,1]), np.mean(face_bgr[:,:,2])

    mean_gray = (mean_b + mean_g + mean_r) / 3.0
    mult_b = np.clip(mean_gray / (mean_b + 1e-6), CLAMP_LO, CLAMP_HI)
    mult_g = np.clip(mean_gray / (mean_g + 1e-6), CLAMP_LO, CLAMP_HI)
    mult_r = np.clip(mean_gray / (mean_r + 1e-6), CLAMP_LO, CLAMP_HI)

    face = face_bgr.astype(np.float32)
    face[:,:,0] = np.clip(face[:,:,0] * mult_b, 0, 255)
    face[:,:,1] = np.clip(face[:,:,1] * mult_g, 0, 255)
    face[:,:,2] = np.clip(face[:,:,2] * mult_r, 0, 255)
    return face.astype(np.uint8)


# --- Exposure Check ---
def check_exposure(face_bgr):
    gray = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2GRAY)
    mean_lum = float(np.mean(gray))
    if mean_lum < 40 or mean_lum > 240:
        return False
    return True


# --- CLAHE + LAB Extraction ---
def extract_lab_pixels(face_bgr):
    """Extract all face pixels in LAB with CLAHE on L channel."""
    lab = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2LAB)
    L, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    L_eq = clahe.apply(L)
    lab_eq = cv2.merge([L_eq, a, b])
    # For pre-cropped faces, use all non-black pixels as "skin"
    mask = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2GRAY)
    mask_bool = mask > 15  # exclude near-black borders
    return lab_eq[mask_bool]


# --- K-Means Undertone Clustering ---
def cluster_undertone(lab_pixels, k=3):
    if len(lab_pixels) < 50:
        return None

    L_vals = lab_pixels[:, 0]
    midtone = (L_vals >= 15) & (L_vals <= 85)
    filtered = lab_pixels[midtone]

    if len(filtered) < 50:
        filtered = lab_pixels

    ab = filtered[:, 1:].astype(np.float32)
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, max_iter=300, random_state=42)
    labels = km.fit_predict(ab)

    counts = np.bincount(labels)
    # Compactness-weighted selection
    scores = np.zeros(k)
    for cid in range(k):
        pts = ab[labels == cid]
        variance = np.mean(np.var(pts, axis=0)) + 1e-6
        scores[cid] = counts[cid] / variance
    dominant_id = int(scores.argmax())

    dominant_ab = km.cluster_centers_[dominant_id]
    dominant_L = np.median(filtered[labels == dominant_id, 0])
    return np.array([dominant_L, dominant_ab[0], dominant_ab[1]])


# --- Color Space Conversion ---
def opencv_lab_to_cie(lab_cv):
    L_cie = lab_cv[0] * 100.0 / 255.0
    a_cie = lab_cv[1] - 128.0
    b_cie = lab_cv[2] - 128.0
    return L_cie, a_cie, b_cie


def calculate_ita(L_cie, b_cie):
    if abs(b_cie) < 1e-6:
        b_cie = 1e-6
    return float(np.degrees(np.arctan2((L_cie - 50.0), b_cie)))


def compute_warmth(a_cie, b_cie):
    return (0.25 * a_cie) + (0.75 * b_cie)


# --- Full Feature Extraction ---
def extract_features(img_bgr):
    """
    Extract all features from a face image.
    Returns: dict with ITA, a_cie, b_cie, L_cie, warmth, confidence
    Or None if extraction fails.
    """
    if img_bgr is None or img_bgr.shape[0] < 50 or img_bgr.shape[1] < 50:
        return None

    # Resize for consistency
    face = cv2.resize(img_bgr, (256, 256), interpolation=cv2.INTER_LANCZOS4)

    if not check_exposure(face):
        return None

    # Grey World (no mask for pre-cropped dataset faces)
    face = apply_grey_world(face)

    # Extract LAB pixels
    lab_pixels = extract_lab_pixels(face)
    if lab_pixels is None or len(lab_pixels) < 100:
        return None

    # K-Means
    dominant = cluster_undertone(lab_pixels, k=3)
    if dominant is None:
        return None

    # Convert to CIE LAB
    L_cie, a_cie, b_cie = opencv_lab_to_cie(dominant)
    ita = calculate_ita(L_cie, b_cie)
    warmth = compute_warmth(a_cie, b_cie)

    # Confidence (cluster tightness)
    ab_pixels = lab_pixels[:, 1:].astype(np.float32)
    dom_ab = dominant[1:].astype(np.float32)
    dists = np.linalg.norm(ab_pixels - dom_ab, axis=1)
    p90 = float(np.percentile(dists, 90))
    confidence = max(0.0, min(1.0, 1.0 - (p90 - 5.0) / 25.0))

    return {
        "ITA": round(ita, 4),
        "a_cie": round(a_cie, 4),
        "b_cie": round(b_cie, 4),
        "L_cie": round(L_cie, 4),
        "warmth": round(warmth, 4),
        "confidence": round(confidence, 4),
    }

print("Feature extraction pipeline ready")

## Step 3: Lumiqe Feature Extraction Pipeline

This is the exact same color science from your backend — extracts ITA, a*, b*, L*, warmth from each face image. No BiSeNet needed here (UTKFace images are already cropped faces).

In [ ]:
## --- Download UTKFace dataset ---
## UTKFace: ~20K face images labeled [age]_[gender]_[race]_[date].jpg
## Race: 0=White, 1=Black, 2=Asian, 3=Indian, 4=Others

import gdown
import zipfile

DATA_DIR = Path("/content/datasets")
DATA_DIR.mkdir(exist_ok=True)

UTKFACE_DIR = DATA_DIR / "UTKFace"

if not UTKFACE_DIR.exists():
    print("Downloading UTKFace dataset...")
    # UTKFace hosted on Google Drive (public mirror)
    url = "https://drive.google.com/uc?id=0BxYys69jI14kYVM3aVhKS1VhRUk"
    zip_path = DATA_DIR / "UTKFace.tar.gz"
    try:
        gdown.download(url, str(zip_path), quiet=False)
        !cd /content/datasets && tar -xzf UTKFace.tar.gz
        print(f"UTKFace extracted to {UTKFACE_DIR}")
    except Exception as e:
        print(f"Auto-download failed: {e}")
        print("\n*** MANUAL DOWNLOAD INSTRUCTIONS ***")
        print("1. Go to: https://susanqq.github.io/UTKFace/")
        print("2. Download 'Aligned&Cropped Faces' (~1.3GB)")
        print("3. Upload the zip to Colab and extract to /content/datasets/UTKFace/")
        print("   OR upload images directly to /content/datasets/UTKFace/")
else:
    print(f"UTKFace already exists at {UTKFACE_DIR}")

# Count images
if UTKFACE_DIR.exists():
    images = list(UTKFACE_DIR.glob("*.jpg")) + list(UTKFACE_DIR.glob("*.png"))
    print(f"Found {len(images)} face images")
else:
    print("WARNING: No images found. Please upload dataset manually.")
    print("You can also upload ANY folder of face images to /content/datasets/faces/")

## Step 2: Download Face Datasets

We use **UTKFace** (20K+ diverse faces with age/gender/ethnicity) and optionally **FairFace** for extra diversity.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("All imports OK")

In [ ]:
!pip install -q scikit-learn==1.8.0 joblib matplotlib seaborn pandas opencv-python-headless mediapipe gdown

## Step 1: Install Dependencies

# Lumiqe Season Classifier — Training Notebook

**Purpose:** Train a real ML model to replace the hardcoded SEASON_MAP lookup table.

**Pipeline:**
1. Download diverse face datasets (UTKFace + FairFace)
2. Run Lumiqe's CV pipeline to extract features (ITA, a*, b*, L*, warmth)
3. Generate pseudo-labels using current SEASON_MAP (bootstrap)
4. Train 3 classifiers (Logistic Regression, Random Forest, MLP)
5. Evaluate with cross-validation + confusion matrix
6. Export best model as `.joblib`
7. Download and integrate with backend

**Runtime:** Use GPU runtime (Runtime → Change runtime type → T4 GPU)